# Vietnamese OCR Dataset
## Exploratory data analysis for a Vietnamese image-to-text recognition model

This notebook covers:
1. Loading and inspecting the data structure
2. Basic dataset statistics
3. Visualizing sample images and their labels
4. Analyzing Vietnamese character and word distributions
5. Checking image properties (dimensions, format)
6. Detecting missing or corrupted data
7. Preprocessing

## 1. Import Required Libraries

In [ ]:
# Standard libraries
import os
import json
import warnings
from pathlib import Path
from collections import Counter

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import seaborn as sns

!apt-get -qq install fonts-dejavu
import matplotlib.pyplot as plt
import matplotlib as mpl

# Configure Matplotlib to use the DejaVu Sans font
mpl.rcParams['font.family'] = 'DejaVu Sans'

# Image processing
from PIL import Image

# Suppress warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_colwidth', None)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("✓ All libraries imported successfully!")

## 2. Load and Parse Label JSON File

Note: before mounting, create a shortcut to the shared Drive inside MyDrive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define paths
DATA_DIR = Path('/content/drive/MyDrive/IntroToML - OCR/tcuong/1')  # The path can be changed freely, but it must point at a folder containing label.json, not the dataset root
LABEL_FILE = DATA_DIR / 'label.json'

# Load label file
with open(LABEL_FILE, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)

# Create DataFrame
df = pd.DataFrame([
    {'image_file': k, 'label': v}
    for k, v in labels_dict.items()
])

# Extract file extension
df['extension'] = df['image_file'].apply(lambda x: x.split('.')[-1].lower())

# Add text length column
df['text_length'] = df['label'].apply(len)

# Add word count column
df['word_count'] = df['label'].apply(lambda x: len(x.split()))

print(f"Total samples: {len(df)}")
print(f"\nFirst 10 samples:")
df.head(10)

## 3. Explore Dataset Statistics

In [ ]:
print(f"\n Basic Statistics:")
print(f"   - Total samples: {len(df)}")
print(f"   - Unique labels: {df['label'].nunique()}")
print(f"   - Duplicate labels: {len(df) - df['label'].nunique()}")

print(f"\n Text Length Statistics:")
print(f"   - Min text length: {df['text_length'].min()} chars")
print(f"   - Max text length: {df['text_length'].max()} chars")
print(f"   - Mean text length: {df['text_length'].mean():.2f} chars")
print(f"   - Median text length: {df['text_length'].median():.0f} chars")
print(f"   - Std text length: {df['text_length'].std():.2f} chars")

print(f"\n Word Count Statistics:")
print(f"   - Min word count: {df['word_count'].min()} words")
print(f"   - Max word count: {df['word_count'].max()} words")
print(f"   - Mean word count: {df['word_count'].mean():.2f} words")

print(f"\n File Format Distribution:")
print(df['extension'].value_counts().to_string())

In [ ]:
# Text length distribution visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Text length histogram
axes[0].hist(df['text_length'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Text Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Text Length')
axes[0].axvline(df['text_length'].mean(), color='red', linestyle='--', label=f'Mean: {df["text_length"].mean():.1f}')
axes[0].legend()

# Word count histogram
axes[1].hist(df['word_count'], bins=20, color='forestgreen', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Word Count')
axes[1].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["word_count"].mean():.1f}')
axes[1].legend()

# File format distribution
df['extension'].value_counts().plot(kind='pie', ax=axes[2], autopct='%1.1f%%', colors=['#66b3ff','#99ff99'])
axes[2].set_title('File Format Distribution')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Visualize Sample Images with Labels

In [ ]:
def display_samples(df, n_samples=9, random_seed=42):
    """Display random sample images with their labels, supporting Vietnamese"""

    # Make sure the configured font supports Vietnamese
    mpl.rcParams['font.family'] = 'DejaVu Sans'

    np.random.seed(random_seed)
    samples = df.sample(n=min(n_samples, len(df)))

    n_cols = 3
    n_rows = (len(samples) + n_cols - 1) // n_cols

    # Slightly larger figsize so Vietnamese titles do not overlap
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 6 * n_rows))

    # Handle the single-sample case
    if n_samples == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for idx, (_, row) in enumerate(samples.iterrows()):
        img_path = DATA_DIR / row['image_file']

        try:
            img = Image.open(img_path)
            axes[idx].imshow(img)

            # Handle the Vietnamese label
            label = str(row['label'])
            if len(label) > 80:
                label = label[:77] + '...'

            # Use wrap=True and tune the font size
            # Add the title using the installed DejaVu font
            axes[idx].set_title(f"File: {row['image_file']}\nLabel: {label}",
                                fontsize=10,
                                fontweight='normal',
                                pad=10,
                                loc='center')
            axes[idx].axis('off')

        except Exception as e:
            axes[idx].text(0.5, 0.5, f"Failed to load file:\n{row['image_file']}",
                          ha='center', va='center', fontsize=10, color='red')
            axes[idx].axis('off')

    # Hide the leftover axes when the sample count is not divisible by n_cols
    for i in range(idx + 1, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Run it
display_samples(df, n_samples=9)

In [ ]:
# Display samples by text length categories
print("Short texts (single words):")
short_texts = df[df['word_count'] <= 2].head(6)
display_samples(short_texts, n_samples=6, random_seed=1)

In [ ]:
print(" Long texts (paragraphs):")
long_texts = df[df['word_count'] > 30].head(3)
display_samples(long_texts, n_samples=3, random_seed=2)

## 5. Analyze Text Label Distribution
### Vietnamese Character & Diacritics Analysis

In [ ]:
# Vietnamese diacritics - special characters in Vietnamese
VIETNAMESE_CHARS = {
    'vowels_with_diacritics': 'àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ',
    'uppercase_vowels': 'ÀÁẢÃẠĂẰẮẲẴẶÂẦẤẨẪẬÈÉẺẼẸÊỀẾỂỄỆÌÍỈĨỊÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢÙÚỦŨỤƯỪỨỬỮỰỲÝỶỸỴ',
    'special_consonant': 'đĐ',
    'basic_chars': 'aăâeêioôơuưybcdfghjklmnpqrstvwxz',
}

# Concatenate all text
all_text = ' '.join(df['label'].tolist())

# Character frequency analysis
char_counter = Counter(all_text.lower())

# Total unique characters
unique_chars = set(all_text)
print(f"\n Total unique characters: {len(unique_chars)}")
print(f"   Unique characters: {''.join(sorted(unique_chars))}")

# Count Vietnamese special characters
vn_special_chars = VIETNAMESE_CHARS['vowels_with_diacritics'] + VIETNAMESE_CHARS['uppercase_vowels'] + VIETNAMESE_CHARS['special_consonant']
vn_char_count = sum(1 for char in all_text if char in vn_special_chars)
total_chars = len(all_text)
print(f"\n🇻🇳 Vietnamese diacritics usage:")
print(f"   - Characters with diacritics: {vn_char_count} ({vn_char_count/total_chars*100:.2f}%)")
print(f"   - Total characters: {total_chars}")

In [ ]:
# Character frequency visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top 30 most common characters
top_chars = char_counter.most_common(30)
chars, counts = zip(*top_chars)
axes[0].barh(range(len(chars)), counts, color='steelblue')
axes[0].set_yticks(range(len(chars)))
axes[0].set_yticklabels([f"'{c}'" if c != ' ' else "'space'" for c in chars])
axes[0].invert_yaxis()
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 30 Most Common Characters')

# Vietnamese diacritics frequency
vn_chars_in_text = [char for char in all_text.lower() if char in VIETNAMESE_CHARS['vowels_with_diacritics']]
vn_char_counter = Counter(vn_chars_in_text)
top_vn_chars = vn_char_counter.most_common(20)

if top_vn_chars:
    vn_chars, vn_counts = zip(*top_vn_chars)
    axes[1].barh(range(len(vn_chars)), vn_counts, color='forestgreen')
    axes[1].set_yticks(range(len(vn_chars)))
    axes[1].set_yticklabels([f"'{c}'" for c in vn_chars])
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Frequency')
    axes[1].set_title('Top 20 Vietnamese Diacritics Characters')

plt.tight_layout()
plt.show()

In [ ]:
# Word frequency analysis
all_words = ' '.join(df['label'].tolist()).split()
word_counter = Counter(all_words)

print("\n Word Frequency Analysis:")
print(f"   - Total words: {len(all_words)}")
print(f"   - Unique words: {len(word_counter)}")
print(f"\n   Top 20 most common words:")
for word, count in word_counter.most_common(20):
    print(f"      '{word}': {count}")

## 6. Check Image Properties (Size, Format, Channels)

In [ ]:
def get_image_properties(image_file):
    """Get properties of an image file"""
    img_path = DATA_DIR / image_file
    try:
        with Image.open(img_path) as img:
            width, height = img.size
            mode = img.mode
            channels = len(img.getbands())
            aspect_ratio = width / height
            return {
                'width': width,
                'height': height,
                'aspect_ratio': aspect_ratio,
                'channels': channels,
                'mode': mode,
                'file_size_kb': os.path.getsize(img_path) / 1024,
                'valid': True
            }
    except Exception as e:
        return {
            'width': None, 'height': None, 'aspect_ratio': None,
            'channels': None, 'mode': None, 'file_size_kb': None,
            'valid': False, 'error': str(e)
        }

# Get properties for all images
print("Analyzing image properties...")
image_props = df['image_file'].apply(get_image_properties)
props_df = pd.DataFrame(image_props.tolist())

# Merge with main dataframe
df_full = pd.concat([df, props_df], axis=1)

# Display statistics
valid_images = df_full[df_full['valid'] == True]
print(f"\n Valid images: {len(valid_images)} / {len(df_full)}")

print(f"\n Image Dimensions:")
print(f"   Width  - Min: {valid_images['width'].min()}, Max: {valid_images['width'].max()}, Mean: {valid_images['width'].mean():.0f}")
print(f"   Height - Min: {valid_images['height'].min()}, Max: {valid_images['height'].max()}, Mean: {valid_images['height'].mean():.0f}")
print(f"   Aspect Ratio - Min: {valid_images['aspect_ratio'].min():.2f}, Max: {valid_images['aspect_ratio'].max():.2f}")

print(f"\n Color Mode Distribution:")
print(valid_images['mode'].value_counts().to_string())

print(f"\n File Size Statistics (KB):")
print(f"   Min: {valid_images['file_size_kb'].min():.2f} KB")
print(f"   Max: {valid_images['file_size_kb'].max():.2f} KB")
print(f"   Mean: {valid_images['file_size_kb'].mean():.2f} KB")

In [ ]:
# Visualize image dimensions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Width distribution
axes[0, 0].hist(valid_images['width'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Width (pixels)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Image Width Distribution')

# Height distribution
axes[0, 1].hist(valid_images['height'], bins=30, color='forestgreen', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Height (pixels)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Image Height Distribution')

# Scatter plot of width vs height
axes[1, 0].scatter(valid_images['width'], valid_images['height'], alpha=0.6, c='steelblue')
axes[1, 0].set_xlabel('Width (pixels)')
axes[1, 0].set_ylabel('Height (pixels)')
axes[1, 0].set_title('Width vs Height Distribution')

# Aspect ratio distribution
axes[1, 1].hist(valid_images['aspect_ratio'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Aspect Ratio (Width/Height)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Aspect Ratio Distribution')

plt.tight_layout()
plt.show()

## 7. Detect Missing or Corrupted Images

In [ ]:
# Check for missing or corrupted images
missing_images = []
corrupted_images = []

for _, row in df.iterrows():
    img_path = DATA_DIR / row['image_file']

    if not os.path.exists(img_path):
        missing_images.append(row['image_file'])
    else:
        try:
            with Image.open(img_path) as img:
                img.verify()  # Verify image integrity
        except Exception as e:
            corrupted_images.append({'file': row['image_file'], 'error': str(e)})

print("=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

print(f"\n Missing images: {len(missing_images)}")
if missing_images:
    for img in missing_images:
        print(f"   - {img}")

print(f"\n Corrupted images: {len(corrupted_images)}")
if corrupted_images:
    for item in corrupted_images:
        print(f"   - {item['file']}: {item['error']}")

# Check for labels without corresponding images
all_image_files = set(os.listdir(DATA_DIR))
all_image_files.discard('label.json')
labeled_files = set(df['image_file'].tolist())

unlabeled_images = all_image_files - labeled_files
orphan_labels = labeled_files - all_image_files

print(f"\n Files without labels: {len(unlabeled_images)}")
if unlabeled_images:
    print(f"   {list(unlabeled_images)[:10]}...")

print(f"\n Labels without files: {len(orphan_labels)}")
if orphan_labels:
    print(f"   {list(orphan_labels)[:10]}...")

if len(missing_images) == 0 and len(corrupted_images) == 0:
    print("\n All images are valid and accessible!")

# Preprocessing

The script below:
+ Mounts the team's data Drive
+ Unzips the three UIT data archives (onto the local Colab disk)
+ Validates every folder and deletes the broken or incomplete ones
+ Copies the result from the local disk back to the team's data Drive
+ Selects samples and builds the Data folder used by the following steps

In [ ]:
!unzip -q "/content/drive/MyDrive/IntroToML - OCR/UIT_HWDB_line.zip" -d "/content/local_data"

In [ ]:
!unzip -q "/content/drive/MyDrive/IntroToML - OCR/UIT_HWDB_word.zip" -d "/content/local_data"

In [ ]:
!unzip -q "/content/drive/MyDrive/IntroToML - OCR/UIT_HWDB_paragraph.zip" -d "/content/local_data"

In [ ]:
import os
import json
import shutil
from pathlib import Path

def strict_cleanup_folders(base_path):
    base_path = Path(base_path)
    deleted_count = 0
    total_folders = 0

    if not base_path.exists():
        print(f"Path does not exist: {base_path}")
        return

    # Walk the numbered subfolders (250, 251, ...)
    for folder in base_path.iterdir():
        if folder.is_dir():
            total_folders += 1
            label_file = folder / 'label.json'

            # 1. Check that label.json exists
            if not label_file.exists():
                print(f"[DELETED] Missing label.json in: {folder.name}")
                shutil.rmtree(folder)
                deleted_count += 1
                continue

            try:
                # 2. Read label.json
                with open(label_file, 'r', encoding='utf-8') as f:
                    labels = json.load(f)

                # Image files actually present in the folder
                actual_images = {f.name for f in folder.glob('*.jpg')}
                # Image files listed in label.json
                labeled_images = set(labels.keys())

                is_valid = True

                # CHECK 1: images on disk that have no label
                extra_images = actual_images - labeled_images
                if extra_images:
                    print(f"[DELETED] {folder.name}: images {list(extra_images)[:3]}... have no label.")
                    is_valid = False

                # CHECK 2: labels with no matching image file
                if is_valid:
                    missing_images = labeled_images - actual_images
                    if missing_images:
                        print(f"[DELETED] {folder.name}: labels for {list(missing_images)[:3]}... have no image file.")
                        is_valid = False

                # Delete the folder if either check failed
                if not is_valid:
                    shutil.rmtree(folder)
                    deleted_count += 1

            except Exception as e:
                print(f"[ERROR] Could not process {folder.name}: {e}")

    print("-" * 30)
    print(f"CLEANUP RESULTS:")
    print(f"- Folders scanned: {total_folders}")
    print(f"- Folders deleted: {deleted_count}")
    print(f"- Valid folders remaining: {total_folders - deleted_count}")




In [ ]:
DATA_PATH = ["/content/local_data/UIT_HWDB_word/train_data", "/content/local_data/UIT_HWDB_word/test_data", "/content/local_data/UIT_HWDB_line/test_data", "/content/local_data/UIT_HWDB_line/train_data", "/content/local_data/UIT_HWDB_paragraph/test_data", "/content/local_data/UIT_HWDB_paragraph/train_data"]

for path in DATA_PATH:
  strict_cleanup_folders(path)